# Notebook 03 — Verbalizer (supervised, simplified)

Train the **verbalizer (AV)**: given an activation `h`, generate a short English description that captures what `h` is about.

```
 h (768)  ──►  [Linear 768→768]  ──►  [Pythia, trainable]  ──►  text z
                   (act_proj)             (autoregressive)
```

### Simplifications vs. the paper

- **Supervised**, not RL: we train AV to predict the first 16 tokens of the snippet that produced `h`. We lose the paper's "AR-verified" guarantee, but keep the autoencoder principle (text as a learned bottleneck for activations).
- **No prompt** wrapping — just `[projected h] → description`.
- **Reuse AR head** from notebook 02 for end-to-end FVE evaluation.

### Pipeline

```
h  ──►  AV  ──►  text z  ──►  AR  ──►  ĥ   →   FVE(h, ĥ)
```

**Requires:**
- `data/activations/pythia160m_layer6_wikitext2_5000.pt` + `.jsonl` (notebook 01)
- `checkpoints/reconstructor_head_baseline.pt` (notebook 02)

## 0. Environment

In [ ]:
# Install deps + one-shot kernel restart (same trick as notebooks 01, 02).
%pip install -q transformers==4.46.0 accelerate matplotlib "sympy>=1.13.3"

import os
MARKER = '/tmp/.kth_nla_kernel_restarted'
if not os.path.exists(MARKER):
    open(MARKER, 'w').close()
    print('Restarting kernel...')
    print('When Colab reconnects, click Runtime > Run all again.')
    os.kill(os.getpid(), 9)
else:
    print('Packages installed; kernel already restarted in this session.')

In [ ]:
import json, random, copy
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# Persistent Drive storage (same as the other notebooks).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/kth-nla-2026')
    print(f'persistent storage: {DATA_ROOT}')
except ImportError:
    DATA_ROOT = Path('.')
    print(f'local storage: {DATA_ROOT.resolve()}')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

## 1. Config

In [ ]:
MODEL_NAME = 'EleutherAI/pythia-160m'
LAYER = 6

ACT_PT      = DATA_ROOT / 'data/activations/pythia160m_layer6_wikitext2_5000.pt'
ACT_META    = DATA_ROOT / 'data/activations/pythia160m_layer6_wikitext2_5000.jsonl'
AR_CKPT     = DATA_ROOT / 'checkpoints/reconstructor_head_baseline.pt'
AV_CKPT_OUT = DATA_ROOT / 'checkpoints/verbalizer.pt'
FIG_DIR     = DATA_ROOT / 'figures';     FIG_DIR.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)

DESC_TOKENS = 16
TRAIN_N, VAL_N, TEST_N = 4000, 500, 500
BATCH_SIZE = 8         # smaller — we're now backpropping through Pythia
EPOCHS = 4
LR = 1e-4              # lower than AR — we're fine-tuning a pretrained model

## 2. Load activations + targets

Targets are the first `DESC_TOKENS` tokens of the snippet that produced each activation — same placeholder we used for the reconstructor's baseline.

In [ ]:
assert ACT_PT.exists(),  f'Missing {ACT_PT}. Run notebook 01 first.'
assert AR_CKPT.exists(), f'Missing {AR_CKPT}. Run notebook 02 first.'

acts = torch.load(ACT_PT, map_location='cpu')
with open(ACT_META, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token

def truncate_to(snippet: str, n: int) -> str:
    ids = tok(snippet, return_tensors='pt', truncation=True, max_length=n).input_ids[0]
    return tok.decode(ids, skip_special_tokens=True)

targets = [truncate_to(r['snippet'], DESC_TOKENS) for r in records]
print(f'loaded {len(acts)} (h, target) pairs')
print('example target:', repr(targets[0]))

## 3. Train/val/test split (same seed as notebook 02 → same splits)

In [ ]:
N = acts.shape[0]
perm = torch.randperm(N, generator=torch.Generator().manual_seed(SEED)).tolist()
split = {
    'train': perm[:TRAIN_N],
    'val':   perm[TRAIN_N:TRAIN_N + VAL_N],
    'test':  perm[TRAIN_N + VAL_N:TRAIN_N + VAL_N + TEST_N],
}

class ActDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        return {'h': acts[j], 'z': targets[j]}

def collate(batch):
    return {'h': torch.stack([b['h'] for b in batch]), 'z': [b['z'] for b in batch]}

loaders = {
    name: DataLoader(ActDataset(ids), batch_size=BATCH_SIZE,
                     shuffle=(name == 'train'), collate_fn=collate)
    for name, ids in split.items()
}

## 4. Two Pythia instances

- `av_base`: a **trainable** copy of Pythia → becomes the verbalizer.
- `ar_base`: a **frozen** copy → the reconstructor's encoder.

In [ ]:
av_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
ar_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device).eval()
for p in ar_base.parameters():
    p.requires_grad_(False)

HIDDEN = av_base.config.hidden_size
print(f'AV is trainable; AR base is frozen. hidden_size={HIDDEN}')

## 5. Verbalizer model

**Input layout** (no prompt — keep it minimal):

```
 position 0:   projected activation (1 "virtual token")
 positions 1+: description tokens during training
```

During training we feed `[act, t0, ..., t_{T-2}]` and predict `[t0, ..., t_{T-1}]` — standard next-token cross-entropy. At inference we feed only `[act]` and decode `DESC_TOKENS` tokens autoregressively.

In [ ]:
class Verbalizer(nn.Module):
    def __init__(self, base_model, hidden: int):
        super().__init__()
        self.base = base_model
        self.embed = base_model.get_input_embeddings()
        self.act_proj = nn.Linear(hidden, hidden)
        # Start from a scaled identity so initial 'activation token' lives roughly
        # in the same regime as real token embeddings (which have norm ~ 0.5–1.0).
        with torch.no_grad():
            self.act_proj.weight.copy_(torch.eye(hidden) * 0.05)
            self.act_proj.bias.zero_()

    def forward(self, h, target_ids, pad_mask):
        """
        h:          (B, D)        — activations
        target_ids: (B, T)        — description tokens
        pad_mask:   (B, T) bool   — True where target is padding
        """
        B, T = target_ids.shape
        act_embed = self.act_proj(h).unsqueeze(1)            # (B, 1, D)
        tgt_embed = self.embed(target_ids)                   # (B, T, D)
        # input: [act, t0, t1, ..., t_{T-2}]  → predicts [t0, ..., t_{T-1}]
        inputs_embeds = torch.cat([act_embed, tgt_embed[:, :-1]], dim=1)  # (B, T, D)
        out = self.base(inputs_embeds=inputs_embeds)
        logits = out.logits                                  # (B, T, V)

        labels = target_ids.clone()
        labels[pad_mask] = -100
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100,
        )
        return loss, logits

    @torch.no_grad()
    def generate(self, h, max_new_tokens: int, temperature: float = 1.0, top_k: int = 50):
        """Sample text autoregressively starting from the activation token."""
        self.eval()
        B, D = h.shape
        act_embed = self.act_proj(h).unsqueeze(1)            # (B, 1, D)
        inputs_embeds = act_embed
        generated = []
        for _ in range(max_new_tokens):
            out = self.base(inputs_embeds=inputs_embeds)
            logits = out.logits[:, -1, :] / max(temperature, 1e-6)
            if top_k:
                v, _ = logits.topk(top_k)
                logits[logits < v[:, -1:].expand_as(logits)] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(-1)  # (B,)
            generated.append(next_id)
            inputs_embeds = torch.cat([inputs_embeds, self.embed(next_id).unsqueeze(1)], dim=1)
        return torch.stack(generated, dim=1)  # (B, max_new_tokens)

av = Verbalizer(av_base, HIDDEN).to(device)
trainable = sum(p.numel() for p in av.parameters() if p.requires_grad)
print(f'AV trainable params: {trainable:,}')

## 6. Train the verbalizer (supervised, next-token CE)

In [ ]:
def encode_targets(texts):
    enc = tok(texts, padding='max_length', truncation=True, max_length=DESC_TOKENS,
              return_tensors='pt')
    target_ids = enc.input_ids.to(device)
    pad_mask = (enc.attention_mask == 0).to(device)
    return target_ids, pad_mask

@torch.no_grad()
def val_loss(loader):
    av.eval()
    total, n = 0.0, 0
    for batch in loader:
        h = batch['h'].to(device)
        target_ids, pad_mask = encode_targets(batch['z'])
        loss, _ = av(h, target_ids, pad_mask)
        total += loss.item() * h.size(0); n += h.size(0)
    return total / n

optim = torch.optim.AdamW([p for p in av.parameters() if p.requires_grad], lr=LR)
history = []

for epoch in range(1, EPOCHS + 1):
    av.train()
    running, steps = 0.0, 0
    for batch in loaders['train']:
        h = batch['h'].to(device)
        target_ids, pad_mask = encode_targets(batch['z'])
        loss, _ = av(h, target_ids, pad_mask)
        optim.zero_grad(); loss.backward(); optim.step()
        running += loss.item(); steps += 1
    train_loss = running / steps
    v_loss = val_loss(loaders['val'])
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': v_loss})
    print(f'epoch {epoch}: train_loss={train_loss:.4f}  val_loss={v_loss:.4f}')

## 7. Sample some descriptions from the trained AV

Qualitative check. We want them to be at least vaguely on-topic for the source snippet.

In [ ]:
N_SHOW = 6
sample_indices = split['test'][:N_SHOW]
sample_h = torch.stack([acts[i] for i in sample_indices]).to(device)
gen_ids = av.generate(sample_h, max_new_tokens=DESC_TOKENS, temperature=1.0, top_k=50)
for k, idx in enumerate(sample_indices):
    src = records[idx]['snippet']
    desc = tok.decode(gen_ids[k], skip_special_tokens=True)
    print(f'--- example {k+1} ---')
    print(f'source snippet (first 80 chars): {src[:80]!r}')
    print(f'AV description                 : {desc!r}')
    print()

## 8. End-to-end pipeline: AV → AR → ĥ → FVE

Rebuild the reconstructor from notebook 02 (frozen base + the saved head), then for each test activation:

1. Sample a description with the trained AV.
2. Run that description through AR.
3. Compare predicted ĥ to true h.

In [ ]:
class Reconstructor(nn.Module):
    """Identical class to notebook 02."""
    def __init__(self, base_model, layer: int, hidden: int):
        super().__init__()
        self.base = base_model
        self.captured = {}
        base_model.gpt_neox.layers[layer].register_forward_hook(self._hook)
        self.head = nn.Linear(hidden, hidden)

    def _hook(self, module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        self.captured['h'] = h

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            _ = self.base(input_ids=input_ids, attention_mask=attention_mask)
        h_seq = self.captured['h']
        last_idx = attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(h_seq.size(0), device=h_seq.device)
        return self.head(h_seq[batch_idx, last_idx])

ar = Reconstructor(ar_base, LAYER, HIDDEN).to(device)
ckpt = torch.load(AR_CKPT, map_location=device)
ar.head.load_state_dict(ckpt['state_dict'])
for p in ar.parameters(): p.requires_grad_(False)
print(f'Loaded AR head from {AR_CKPT}; baseline test FVE was {ckpt["test_metrics"]["fve"]:+.4f}')

In [ ]:
def fve(h_pred, h_true):
    mse = ((h_pred - h_true) ** 2).sum(dim=1).mean()
    var = ((h_true - h_true.mean(0)) ** 2).sum(dim=1).mean()
    return (1.0 - mse / var).item()

@torch.no_grad()
def evaluate_pipeline(loader, temperature=1.0, top_k=50):
    av.eval(); ar.eval()
    preds, trues, descs = [], [], []
    for batch in loader:
        h = batch['h'].to(device)
        gen_ids = av.generate(h, max_new_tokens=DESC_TOKENS,
                              temperature=temperature, top_k=top_k)
        gen_texts = [tok.decode(g, skip_special_tokens=True) for g in gen_ids]
        enc = tok(gen_texts, padding=True, truncation=True, max_length=DESC_TOKENS,
                  return_tensors='pt').to(device)
        h_hat = ar(enc.input_ids, enc.attention_mask).cpu()
        preds.append(h_hat); trues.append(batch['h']); descs.extend(gen_texts)
    p = torch.cat(preds); t = torch.cat(trues)
    return {
        'fve': fve(p, t),
        'cos': F.cosine_similarity(p, t, dim=1).mean().item(),
        'descs': descs,
    }

test_result = evaluate_pipeline(loaders['test'])
print('====  TEST (AV → AR pipeline)  ====')
print(f"  FVE  = {test_result['fve']:+.4f}")
print(f"  cos  = {test_result['cos']:+.4f}")
print(f"  baseline AR-on-placeholder FVE was {ckpt['test_metrics']['fve']:+.4f}")

## 9. Save the trained verbalizer

In [ ]:
torch.save({
    'state_dict': {k: v.cpu() for k, v in av.state_dict().items()},
    'config': {
        'model_name': MODEL_NAME, 'layer': LAYER, 'hidden': HIDDEN,
        'desc_tokens': DESC_TOKENS, 'epochs': EPOCHS, 'lr': LR, 'seed': SEED,
    },
    'test_pipeline': {k: v for k, v in test_result.items() if k != 'descs'},
    'sample_descs': test_result['descs'][:20],
    'history': history,
}, AV_CKPT_OUT)
print('saved', AV_CKPT_OUT)

## 10. (Optional) plot training curves

In [ ]:
import matplotlib.pyplot as plt
epochs = [h['epoch'] for h in history]
plt.figure(figsize=(6, 3.5))
plt.plot(epochs, [h['train_loss'] for h in history], label='train')
plt.plot(epochs, [h['val_loss']   for h in history], label='val')
plt.xlabel('epoch'); plt.ylabel('next-token CE loss')
plt.title('Verbalizer training')
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / '03_verbalizer_training.png', dpi=120)
plt.show()